In [1]:
import os
# from langchain_community.vectorstores.qdrant import Qdrant
from langchain_qdrant.qdrant import QdrantClient
from langchain_openai.embeddings import OpenAIEmbeddings
import qdrant_client
from dotenv import load_dotenv, find_dotenv

load_dotenv(find_dotenv())

True

In [3]:
# create a qdrant client
os.environ["QDRANT_HOST"] = "https://ee631959-51a5-4aa3-b502-9a1cc6c6031f.us-east-1-0.aws.cloud.qdrant.io:6333"
os.environ["QDRANT_API_KEY"] = os.getenv("QDRANT_API_KEY")
os.environ["QDRANT_COLLECTION"] = "my-collection"

In [4]:
# define the client
client = qdrant_client.QdrantClient(
    url=os.environ["QDRANT_HOST"],
    api_key=os.environ["QDRANT_API_KEY"],
)

In [6]:
client.delete_collection(collection_name=os.environ["QDRANT_COLLECTION"])

True

In [7]:
# create a collection
created_collection = client.create_collection(
    collection_name=os.environ["QDRANT_COLLECTION"],
    vectors_config=qdrant_client.http.models.VectorParams(
        size=1536,  # size of the embedding vector for OpenAI
        distance=qdrant_client.http.models.Distance.COSINE,  # distance metric
    ),
)

In [ ]:
doc_store = QdrantClient(
    client=client,
    collection_name=os.environ["QDRANT_COLLECTION"],
    embeddings=OpenAIEmbeddings(api_key=os.environ["OPENAI_API_KEY"]),
)

C:\Users\Dikshant\AppData\Local\Temp\ipykernel_3148\2508911868.py:1: LangChainDeprecationWarning: The class `Qdrant` was deprecated in LangChain 0.0.37 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-qdrant package and should be used instead. To use it run `pip install -U :class:`~langchain-qdrant` and import as `from :class:`~langchain_qdrant import Qdrant``.
  doc_store = Qdrant(


In [8]:
with open("naruto_story.txt", "r", encoding="utf-8") as f:
    naruto_story = f.read()

In [9]:
naruto_story

'Naruto: The Legacy of the Seventh Hokage \n\nThe Hidden Leaf Village was at peace, but Naruto Uzumaki, the Seventh Hokage, felt an unease stirring in his heart. It had been years since the Fourth Great Ninja War, and though the world had healed, shadows of the past never truly disappeared.  \n\nOne evening, as Naruto overlooked the Hokage Monument, a masked figure appeared before him. “Seventh Hokage,” the stranger spoke, his voice carrying the weight of forgotten history. “You have lived in the light for too long. But do you know what lurks beneath?”  \n\nBefore Naruto could respond, the figure vanished, leaving behind a parchment sealed with an ancient Uzumaki clan symbol. Curious and cautious, Naruto took the scroll back to his office. As he unraveled it, his eyes widened in disbelief. It contained a message from the First Hokage, Hashirama Senju.  \n\n*"The true origin of chakra is not what you have been told. Seek the ruins of the Forgotten Temple beyond the Land of Fire. There, 

In [10]:
from langchain_text_splitters import CharacterTextSplitter
# Split the text into chunks
text_splitter = CharacterTextSplitter(
    separator="\n",
	chunk_size=1000,
	chunk_overlap=200,
	length_function=len
)
chunks = text_splitter.split_text(naruto_story)

In [11]:
chunks

['Naruto: The Legacy of the Seventh Hokage \nThe Hidden Leaf Village was at peace, but Naruto Uzumaki, the Seventh Hokage, felt an unease stirring in his heart. It had been years since the Fourth Great Ninja War, and though the world had healed, shadows of the past never truly disappeared.  \nOne evening, as Naruto overlooked the Hokage Monument, a masked figure appeared before him. “Seventh Hokage,” the stranger spoke, his voice carrying the weight of forgotten history. “You have lived in the light for too long. But do you know what lurks beneath?”  \nBefore Naruto could respond, the figure vanished, leaving behind a parchment sealed with an ancient Uzumaki clan symbol. Curious and cautious, Naruto took the scroll back to his office. As he unraveled it, his eyes widened in disbelief. It contained a message from the First Hokage, Hashirama Senju.',
 '*"The true origin of chakra is not what you have been told. Seek the ruins of the Forgotten Temple beyond the Land of Fire. There, you wi

In [12]:
# add the chunks to the vector store
doc_store.add_texts(chunks)

['17810897ca06435c9dfbac1cfffdf867',
 '8905aedae0594aa28a2fc9c95d6a645d',
 '524a4d42362349f0919a9d6b1f0b52ac']

In [14]:
client.upsert(
    collection_name=os.environ["QDRANT_COLLECTION"],
    points=[
        qdrant_client.http.models.PointStruct(
            id=i,
            vector=doc_store.embeddings.embed_query(chunk),
            payload={"text": chunk},
        )
        for i, chunk in enumerate(chunks)
    ],
)

UpdateResult(operation_id=1, status=<UpdateStatus.COMPLETED: 'completed'>)

In [ ]:
retriever = doc_store.as_retriever(search_kwargs={"k": 3})

In [ ]:
from langchain_google_genai.chat_models import ChatGoogleGenerativeAI

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser
# Define a prompt template
prompt_template = """Answer the following question based on the provided context:

Context: {context}

Question: {query}

Answer: """

prompt = ChatPromptTemplate.from_template(prompt_template)

# Create the LCEL chain
retriever_chain = (
    {"context": retriever, "query": RunnablePassthrough()} 
    | prompt 
    | ChatGoogleGenerativeAI(temperature=0, model="gemini-2.0-flash")
    | StrOutputParser()
)

# Invoke the chain
retriever_chain.invoke("What is the name of the village where Naruto was born?")

In [ ]:
from google import genai

client = genai.Client()
response = client.models.embed_content(
    model="text-embedding-005",
    contents=[
        "How do I get a driver's license/learner's permit?",
        "How do I renew my driver's license?",
        "How do I change my address on my driver's license?",
    ]
)
print(response)

In [7]:
import google.generativeai as gemini_client
from qdrant_client import QdrantClient

gemini_client.configure(api_key=os.environ["GOOGLE_API_KEY"])
texts = ["Example text 1", "Example text 2"]
results = [gemini_client.embed_content(model="models/embedding-001", content=text, task_type="retrieval_document") for text in texts]


In [16]:
results[0]['embedding']

[0.021951523,
 -0.033398803,
 -0.06551014,
 -0.00520707,
 0.083434835,
 0.039356984,
 0.03170321,
 -0.038868092,
 -0.004334161,
 0.046912495,
 0.004956529,
 0.034559157,
 -0.0118518155,
 0.031592995,
 -0.00071801303,
 -0.028148802,
 0.031615946,
 0.026078887,
 -0.0019419874,
 -0.015882395,
 0.018656358,
 0.0062244637,
 0.0045536347,
 -0.013240311,
 0.03656951,
 -0.0035492366,
 0.0043982468,
 -0.059552316,
 -0.045446023,
 0.0116651915,
 -0.028638765,
 -0.00071613735,
 -0.060094062,
 0.014566309,
 0.011892311,
 -0.04576237,
 -0.019438595,
 0.010469582,
 -0.025278106,
 0.018409645,
 -0.01022336,
 -0.016770402,
 -0.021899676,
 0.014210052,
 0.016316107,
 -0.00623331,
 -0.035729773,
 0.043169737,
 -0.00085528183,
 -0.033324514,
 0.041471034,
 -0.0027753164,
 0.07010081,
 -0.047064643,
 0.01680804,
 -0.05081854,
 0.0424126,
 -0.00866079,
 -0.010386974,
 0.0049481303,
 -0.017149534,
 0.016820963,
 0.026358005,
 -0.0016168265,
 -0.06773892,
 -0.08692613,
 -0.06335963,
 0.003555085,
 0.05004031

In [18]:
results[1]['embedding']

[0.023851352,
 -0.03194138,
 -0.06164465,
 -0.005315997,
 0.080973126,
 0.032837834,
 0.02625478,
 -0.038818453,
 -0.0087797195,
 0.054468516,
 0.0017100933,
 0.031582016,
 -0.0106940605,
 0.035735354,
 -0.0015799721,
 -0.02195878,
 0.036663562,
 0.029269917,
 0.0008563768,
 -0.012799886,
 0.020647202,
 -0.001733922,
 0.0065054093,
 -0.00804344,
 0.032704186,
 -0.003911399,
 0.002556915,
 -0.061169,
 -0.05304298,
 0.007035714,
 -0.03199952,
 -0.0074435174,
 -0.062243707,
 0.014960303,
 0.010111896,
 -0.042725146,
 -0.0169535,
 0.009329407,
 -0.021808824,
 0.015064822,
 -0.008757686,
 -0.018266039,
 -0.02050868,
 0.015753772,
 0.016144644,
 0.0011244426,
 -0.037338953,
 0.043756995,
 -0.0069814753,
 -0.030996082,
 0.04314769,
 0.0023470616,
 0.06580489,
 -0.046605017,
 0.012161406,
 -0.053668864,
 0.03787357,
 -0.006561757,
 -0.010400619,
 0.0044761626,
 -0.01454057,
 0.019264035,
 0.026039146,
 -0.002736243,
 -0.07360666,
 -0.089242764,
 -0.06674818,
 0.0030033595,
 0.04286842,
 -0.000